# Reading and writing JDFTx files with pymatgen

This notebook is a hands-on tour of `pymatgen.io.jdftx`. You'll:

1. read a JDFTx input file and look at its settings,
2. change settings, and see how mistakes are caught,
3. build an input file from a pymatgen `Structure`,
4. read the results of a run: structure, energy, forces,
5. follow the steps of a geometry optimization,
6. read atomic charges and magnetic moments,
7. write an input file that restarts a run,
8. read eigenvalues and band projections from dump files.

Everything uses pymatgen's JDFTx test files, so you don't need to run JDFTx. For now the notebook reads them from the
`test-files/io/jdftx` folder of the pymatgen-core repository: run it from its own folder (`jdftx_user_guide/`) in a
clone of the repository, with pymatgen installed from that clone (`pip install -e .`).

> **Draft.** Written against pymatgen-core's `misc_fixes` branch. A few outputs will change as open issues are fixed;
> see `PLAN.md`.

In [ ]:
from __future__ import annotations

from pathlib import Path

from pymatgen.core import Lattice, Structure
from pymatgen.io.jdftx.inputs import JDFTXInfile
from pymatgen.io.jdftx.outputs import JDFTXOutfile, JDFTXOutputs

# The example files: pymatgen-core's JDFTx test files, relative to this notebook's folder
EXAMPLES = Path("../test-files/io/jdftx")
IN_FILES = EXAMPLES / "test_jdftx_in_files"
OUT_FILES = EXAMPLES / "test_jdftx_out_files"
CALC_DIRS = EXAMPLES / "test_jdftx_calc_dirs"

## 1. Reading an input file

`JDFTXInfile` reads a JDFTx input file into a dictionary. Each key is a JDFTx command, and its value is that command's
setting. Here is the example file, a water molecule in implicit water with dissolved NaF:

In [ ]:
print((IN_FILES / "h2o.in").read_text())

In [ ]:
jif = JDFTXInfile.from_file(IN_FILES / "h2o.in")
sorted(jif)

Settings come in three forms:

- a single value, for commands that take one value, like `elec-ex-corr`;
- a dictionary of named parameters, for commands that take several, like `elec-cutoff`;
- a list, for commands that can be given more than once, like `ion` and `fluid-cation`. Such a command is a list even
  when the file gives it only once.

In [ ]:
print(jif["elec-ex-corr"])
print(jif["elec-cutoff"])
print(jif["fluid-cation"])
print(jif["ion"][0])

The parameter names are JDFTx's own (see `jdftx -t` or the JDFTx command reference); values are in JDFTx's units, so
`elec-cutoff` is in Hartree and the `ion` positions are in bohr.

The lattice and ions are also available as a pymatgen `Structure`, converted to Å:

In [ ]:
jif.structure

## 2. Changing settings

Set a command like a dictionary entry, add another line of a repeatable command with `append_tag`, and remove a command
with `del`:

In [ ]:
jif["elec-cutoff"] = {"Ecut": 25, "EcutRho": 125}
jif["elec-smearing"] = {"smearingType": "Fermi", "smearingWidth": 0.01}
jif.append_tag("fluid-cation", {"name": "K+", "concentration": 0.1})
jif.append_tag("fluid-anion", {"name": "Cl-", "concentration": 0.1})
del jif["ionic-minimize"]  # a single-point calculation instead of a geometry optimization

A value can also be given as it would appear in the file. pymatgen reads it with JDFTx's rules, so a mistake is caught
here rather than when JDFTx runs:

In [ ]:
try:
    jif["elec-smearing"] = "Fermi-Dirac 0.01"
except ValueError as error:
    print(error)

`print(jif)` shows the file pymatgen will write, and `write_file` writes it:

In [ ]:
print(jif)
jif.write_file("h2o_edited.in")

## 3. Starting from a pymatgen Structure

`JDFTXInfile.from_structure` turns a `Structure` into JDFTx's `lattice` and `ion` commands, converting Å to bohr. Add
the other settings with `+`, which combines two input files:

In [ ]:
a = 5.43  # silicon's lattice constant, in Å
fcc = Lattice([[0, a / 2, a / 2], [a / 2, 0, a / 2], [a / 2, a / 2, 0]])
silicon = Structure(fcc, ["Si", "Si"], [[0, 0, 0], [0.25, 0.25, 0.25]])

settings = JDFTXInfile.from_dict(
    {
        "ion-species": "GBRV/$ID_pbe.uspp",
        "elec-ex-corr": "gga-PBE",
        "elec-cutoff": {"Ecut": 20, "EcutRho": 100},
        "kpoint-folding": {"n0": 4, "n1": 4, "n2": 4},
        "elec-smearing": {"smearingType": "Fermi", "smearingWidth": 0.01},
    }
)
si_input = JDFTXInfile.from_structure(silicon) + settings
print(si_input)
si_input.write_file("si.in")

`from_structure` also takes per-atom `selective_dynamics` (which atoms may move) and `velocities`. Atoms whose JDFTx
species isn't named after its element (for example `Fe1` and `Fe2`, to give the two kinds of iron in an antiferromagnet
different pseudopotential files) keep that name in the `jdftx_species` site property.

## 4. Reading the results of a run

`JDFTXOutfile` reads the output file JDFTx writes (the file given to `jdftx -o`). pymatgen converts to its usual
units: energies in eV, lengths in Å, forces in eV/Å. This example is a short geometry optimization of H₂:

In [ ]:
out = JDFTXOutfile.from_file(OUT_FILES / "eopt_scf_ionmin.out")
print(out.structure)
print("Energy (eV):", out.e)
print("Forces (eV/Å):")
print(out.forces)

In [ ]:
out.ecomponents  # the energy broken into its terms, in eV

If JDFTx was started several times with the same output file, the file holds several runs. Each is one entry of
`out.slices`, and the attributes of `out` itself describe the last one:

In [ ]:
len(out.slices)

## 5. Following a geometry optimization

`out.jstrucs` holds the structure of each ionic step, with its own energy and forces:

In [ ]:
for i, step in enumerate(out.jstrucs):
    print(f"step {i}: energy {step.e:.6f} eV, largest force {abs(step.forces).max():.4f} eV/Å")

`out.trajectory` gives the same steps as a pymatgen `Trajectory`, for example to write an animation:

In [ ]:
out.trajectory

## 6. Atomic charges and magnetic moments

JDFTx's Lowdin analysis gives each atom a charge and, in spin-polarized runs, a magnetic moment. pymatgen stores them as
site properties of the structure. This example is a spin-polarized, stretched H₂ whose two atoms were given different
species names, `H1` and `H2`:

In [ ]:
spin_run = JDFTXOutfile.from_file(OUT_FILES / "species_names.out")
properties = spin_run.structure.site_properties
print("species names:", properties["jdftx_species"])
print("charges:", properties["charges"])
print("magnetic moments:", properties["magmom"])

These are the values at the final step, where the stretched molecule has lost its spin. Each step in `spin_run.jstrucs`
has its own; at the first step the two atoms still carry opposite moments:

In [ ]:
print("first-step magnetic moments:", spin_run.jstrucs[0].site_properties["magmom"])

## 7. Restarting from a finished run

`to_jdftxinfile()` builds an input file that continues from the run's final state: the same settings, with the final
positions (and, for molecular dynamics, the final velocities).

In [ ]:
restart = out.to_jdftxinfile()
restart.write_file("restart.in", strip_nan=True)
print(Path("restart.in").read_text()[:500])

> JDFTx prints some unset values as `nan`, which it can't read back. `strip_nan=True` leaves them out. (This should
> become the default; see `PLAN.md`.)

## 8. Eigenvalues and band projections

`JDFTXOutputs` reads a whole calculation folder: the output file plus the binary dump files JDFTx writes, such as
`eigenvals` and `bandProjections`. Say which dump files to load with `store_vars`:

In [ ]:
outputs = JDFTXOutputs.from_calc_dir(CALC_DIRS / "H2_nbands_multiplier", store_vars=["eigenvals", "bandProjections"])
print("eigenvalues (eV), one row per state:", outputs.eigenvals)
print("projection orbitals:", outputs.orb_label_list)
print("projections array shape (states, bands, orbitals):", outputs.bandProjections.shape)

## Where to go next

- `jdftx_io_user_guide.md` covers each part in more detail: how every kind of setting is represented, units and
  coordinate systems, restarts, and known limitations.
- The JDFTx command reference lists every command and parameter: <https://jdftx.org/Commands.html>.